<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task6/Task6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
#import and start pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
os.listdir('/content/drive/MyDrive/')

['TASK_DATASET', 'Colab Notebooks']

In [17]:
os.listdir('/content/drive/MyDrive/TASK_DATASET/')

['epd_snomed_202603.csv',
 'train_data.parquet',
 'test_data.parquet',
 'results_dt.json',
 'results_logreg.json',
 'results_lr.json',
 'results_rf.json',
 'task3_summary_table.csv',
 'roc_pr_curves.png',
 'shap_summary.png',
 'shap_lr_summary.png',
 'shap_dt_summary.png',
 'shap_logreg_summary.png',
 'tableau_model_summary.csv',
 'tableau_data_quality.csv',
 'tableau_feature_importance.csv',
 'tableau_regional_summary.csv',
 'tableau_scalability.csv']

In [18]:
#READ CSV FILE USING SPARK
df = spark.read.csv('/content/drive/MyDrive/TASK_DATASET/epd_snomed_202603.csv', header=True, inferSchema=True)

In [19]:
from pyspark.sql.functions import when, col

df = df.withColumn('HIGH_COST', when(col('ACTUAL_COST') > 50, 1).otherwise(0))

In [21]:
# Task 6 - Export data for Tableau

# 1. Export model summary metrics for Dashboard 2
import pandas as pd

model_summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Logistic Regression', 'Decision Tree'],
    'Type': ['Regression', 'Regression', 'Classification', 'Classification'],
    'RMSE': [8.23, 115.21, None, None],
    'R2': [0.9988, 0.7400, None, None],
    'MAE': [2.43, 19.40, None, None],
    'Accuracy': [None, None, 0.9720, 0.9929],
    'AUC_ROC': [None, None, 0.9983, 0.9974],
    'F1_Score': [None, None, 0.9712, 0.9929],
    'Training_Time_Seconds': [42.91, 44.76, 49.69, 82.67]
})

# Tableau Public works with CSV files, not Parquet
model_summary.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_model_summary.csv', index=False)
print("Model summary exported!")
print(model_summary)

Model summary exported!
                 Model            Type    RMSE      R2    MAE  Accuracy  \
0    Linear Regression      Regression    8.23  0.9988   2.43       NaN   
1        Random Forest      Regression  115.21  0.7400  19.40       NaN   
2  Logistic Regression  Classification     NaN     NaN    NaN    0.9720   
3        Decision Tree  Classification     NaN     NaN    NaN    0.9929   

   AUC_ROC  F1_Score  Training_Time_Seconds  
0      NaN       NaN                  42.91  
1      NaN       NaN                  44.76  
2   0.9983    0.9712                  49.69  
3   0.9974    0.9929                  82.67  


In [22]:
#Show null counts, data distributions, partition sizes, ingestion timing

# Data quality metrics from Task 1/2

data_quality = pd.DataFrame({

    'Metric': [

        'Total Rows (Original)', 'Total Columns (Original)',

        'Columns After Cleaning', 'Missing POSTCODE',

        'Missing SNOMED_CODE', 'Rows After Cleaning',

        'File Size GB', 'Partitions Used'

    ],

    'Value': [18364409, 27, 19, 6793, 11449, 18364409, 7.15, 8]

})



data_quality.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_data_quality.csv', index=False)

print("Data quality metrics exported!")

print(data_quality)

Data quality metrics exported!
                     Metric        Value
0     Total Rows (Original)  18364409.00
1  Total Columns (Original)        27.00
2    Columns After Cleaning        19.00
3          Missing POSTCODE      6793.00
4       Missing SNOMED_CODE     11449.00
5       Rows After Cleaning  18364409.00
6              File Size GB         7.15
7           Partitions Used         8.00


In [23]:
#Show model comparison metrics, ROC/AUC chart, SHAP feature importance rankings

# Random Forest feature importance from Task 3

feature_importance = pd.DataFrame({

    'Feature': ['NIC', 'LOG_NIC', 'ITEMS', 'TOTAL_QUANTITY', 'QUANTITY',

                'SNOMED_CODE', 'LOG_TOTAL_QUANTITY', 'LOG_QUANTITY', 'ADQ_USAGE'],

    'Importance': [0.3914, 0.1925, 0.1294, 0.0433, 0.0410, 0.0248, 0.0201, 0.0080, 0.0037]

})



feature_importance.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_feature_importance.csv', index=False)

print("Feature importance exported!")

Feature importance exported!


In [24]:
#Turn data findings into actionable recommendations; connect predictions to real-world decisions
# Aggregate actual NHS data by region for business insights dashboard

import pyspark.sql.functions as F

# Aggregate actual NHS data by region for business insights dashboard

# Use FULL dataset for regional business insights - this matters for credibility
regional_summary = df.groupBy('REGIONAL_OFFICE_NAME').agg(
    F.avg('ACTUAL_COST').alias('avg_cost'),
    F.sum('ACTUAL_COST').alias('total_cost'),
    F.count('*').alias('prescription_count'),
    F.sum('HIGH_COST').alias('high_cost_count')
).toPandas()

regional_summary.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_regional_summary.csv', index=False)
print("Regional summary exported (FULL dataset - 18.3M rows)!")
print(regional_summary)

Regional summary exported (FULL dataset - 18.3M rows)!
       REGIONAL_OFFICE_NAME   avg_cost    total_cost  prescription_count  \
0           EAST OF ENGLAND  51.846550  1.030415e+08             1987432   
1                NORTH WEST  46.934153  1.285601e+08             2739158   
2                    LONDON  41.432403  1.150546e+08             2776923   
3                SOUTH EAST  55.595339  1.417955e+08             2550492   
4  NORTH EAST AND YORKSHIRE  51.855102  1.531392e+08             2953214   
5                  MIDLANDS  51.151925  1.865328e+08             3646643   
6                SOUTH WEST  52.646566  8.979651e+07             1705648   
7              UNIDENTIFIED  63.301853  3.101158e+05                4899   

   high_cost_count  
0           385213  
1           502013  
2           462377  
3           519234  
4           575912  
5           702983  
6           341437  
7             1221  


In [25]:
import pandas as pd

# Training time vs data complexity for scalability dashboard
# Using Task 3 results

scalability_data = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Logistic Regression', 'Decision Tree'],
    'Sample_Size': [44001, 44001, 44001, 44001],
    'Training_Time_Sec': [42.91, 44.76, 49.69, 82.67],
    'Param_Combinations': [1, 1, 4, 4],
    'Driver_Memory_GB': [8, 8, 8, 8],
    'Partitions': [8, 8, 8, 8]
})

# Save to your Drive
scalability_data.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_scalability.csv', index=False)
print("Scalability data exported!")
print(scalability_data)

Scalability data exported!
                 Model  Sample_Size  Training_Time_Sec  Param_Combinations  \
0    Linear Regression        44001              42.91                   1   
1        Random Forest        44001              44.76                   1   
2  Logistic Regression        44001              49.69                   4   
3        Decision Tree        44001              82.67                   4   

   Driver_Memory_GB  Partitions  
0                 8           8  
1                 8           8  
2                 8           8  
3                 8           8  
